# Setup and Load Data

In [0]:
from pyspark.sql.functions import (
    to_timestamp, substring, lpad, concat,
    year, month, dayofmonth, hour, make_timestamp, 
    col, count, when, isnan, avg, sum as spark_sum, lit, desc,
    min, max, mean, stddev, log1p, broadcast,
    regexp_replace, try_to_timestamp, date_trunc, expr, least
)

import pyspark.sql.functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import DataFrame
from graphframes import GraphFrame
from functools import reduce

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

# Read in 

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
#df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
otpw_full_df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
weather_full = spark.read.parquet('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')

In [0]:
def get_size_bytes(path):
    total = 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            total += get_size_bytes(f.path)
        else:
            total += f.size
    return total


def get_size_gb(path):
    return get_size_bytes(path) / (1024**3)

otpw_full_size = get_size_gb(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
weather_full_size = get_size_gb('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')

otpw_full_size, weather_full_size

# Rejoin OTPW and Weather DF for weather to be matched with flight 2 hours before departure (to prevent leakage)

In [0]:
def rejoin_otpw_and_weather(otpw_df, weather_df = None, hrs_before_departure = 2):
    '''Rejoin OTPW and Weather DF such that recorded weather observations are at least the specified number of hours before a flight's scheduled departure.
    Default # of hours is set to 2.
    '''
    if weather_df is None:
        weather_df = spark.read.parquet('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')

    # filter on fm-15
    df_weather_filtered = weather_df.filter(col('REPORT_TYPE') == 'FM-15')   

    stations_set = {
        row["origin_station_id"]
        for row in otpw_df.select("origin_station_id").distinct().collect()
    }

    # filter on only stations in otpw
    df_weather_filtered = df_weather_filtered.filter(col('station').isin(stations_set))
    
    # Convert to timestamp. Create "ts" from "DATE"
    df_weather_filtered = df_weather_filtered.withColumn(
        "ts",
        to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss")
    )

    df_weather_filtered = df_weather_filtered.filter(col("ts").isNotNull())

    # Truncate to hour. Create hour_trunc from "ts"
    df_weather_filtered = df_weather_filtered.withColumn(
        f"{hrs_before_departure}_hour_trunc",
        date_trunc("hour", col("ts"))
    )

    # List of columns to keep
    cols_to_keep = [
        'STATION',
        'DATE',
        'REPORT_TYPE',
        'SOURCE',
        'HourlyAltimeterSetting',
        'HourlyDewPointTemperature',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyPresentWeatherType',
        'HourlyPressureChange',
        'HourlyPressureTendency',
        'HourlyRelativeHumidity',
        'HourlySkyConditions',
        'HourlySeaLevelPressure',
        'HourlyStationPressure',
        'HourlyVisibility',
        'HourlyWetBulbTemperature',
        'HourlyWindDirection',
        'HourlyWindGustSpeed',
        'HourlyWindSpeed',
        f'{hrs_before_departure}_hour_trunc'
    ]

    # Filter the DataFrame to only these columns
    df_weather_filtered = df_weather_filtered.select(*cols_to_keep)

    # In df_weather_filtered, rename STATION → origin_station_id to match OTPW
    df_weather_filtered = df_weather_filtered.withColumnRenamed("STATION", "origin_station_id")

    weather_cols = [
        'HourlyAltimeterSetting',
        'HourlyDewPointTemperature',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyPresentWeatherType',
        'HourlyPressureChange',
        'HourlyPressureTendency',
        'HourlyRelativeHumidity',
        'HourlySkyConditions',
        'HourlySeaLevelPressure',
        'HourlyStationPressure',
        'HourlyVisibility',
        'HourlyWetBulbTemperature',
        'HourlyWindDirection',
        'HourlyWindGustSpeed',
        'HourlyWindSpeed'
    ]

    df_weather_renamed = df_weather_filtered
    for col_name in weather_cols:
        df_weather_renamed = df_weather_renamed.withColumnRenamed(col_name, f"{hrs_before_departure}h_{col_name}")
        print(f"{hrs_before_departure}h_{col_name}")
        assert(f'{hrs_before_departure}h_{col_name}' in df_weather_renamed.columns)

    # drop duplicates
    df_weather_renamed = df_weather_renamed.dropDuplicates(["origin_station_id", f'{hrs_before_departure}_hour_trunc'])

    # Drop columns that are no longer needed (original hourly weather vars and DATE)
    df_weather_renamed = df_weather_renamed.drop(*weather_cols)
    df_weather_renamed = df_weather_renamed.drop('DATE')
    otpw_df = otpw_df.drop(col('REPORT_TYPE'), col('SOURCE'))

    # Create "sched_depart_date_time" column, the scheduled flight departure in local time.
    if 'sched_depart_date_time' not in otpw_df.columns:
        print("Creating sched_depart_date_time in otpw")
        otpw_df = otpw_df.withColumn(
            "sched_depart_date_time",
            make_timestamp(
                col("YEAR"),             # year
                col("MONTH"),         # month
                col("DAY_OF_MONTH"),        # day
                substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 1, 2),  # hour
                substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 3, 2),  # minute
                col("sched_depart_date_time_UTC").substr(15, 2)      # seconds (usually "00")
            )
        )

    # Create "actual_depart_date_time" column, the actual flight departure in local time.
    if 'actual_depart_date_time' not in otpw_df.columns:
        print("Creating actual_depart_date_time in otpw")
        otpw_df = otpw_df.withColumn(
            "actual_depart_date_time",
            make_timestamp(
                col("YEAR"),             # year
                col("MONTH"),         # month
                col("DAY_OF_MONTH"),        # day
                substring(lpad(col("DEP_TIME"), 4, "0"), 1, 2),  # hour
                substring(lpad(col("DEP_TIME"), 4, "0"), 3, 2),  # minute
                lit("00")      # seconds (usually "00")
            )
        )

    # Optional safety
    otpw_df = otpw_df.filter(col("sched_depart_date_time").isNotNull())
    otpw_df = otpw_df.filter(col("actual_depart_date_time").isNotNull())
    otpw_df = otpw_df.withColumn(f"{hrs_before_departure}_hour_trunc", date_trunc("hour", least(col("sched_depart_date_time"), col("actual_depart_date_time")) - expr(f"INTERVAL {hrs_before_departure} HOURS")))

    otpw_df_joined = otpw_df.join(
        df_weather_renamed,
        on=["origin_station_id", f"{hrs_before_departure}_hour_trunc"],  # join keys
        how="left"
    )

    return otpw_df_joined

def get_dimensions(df):
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")

In [0]:
# Match weather observations 2h and 6h before the earlier of each flight's scheduled or actual departure time
otpw_full_2h_6h = rejoin_otpw_and_weather(rejoin_otpw_and_weather(otpw_full_df, weather_full), hrs_before_departure=6)

get_dimensions(otpw_full_df)
get_dimensions(otpw_full_2h_6h)

There are 470k (1.5%) less rows in the joined dataset due to having to filter out null values for the actual departing date time. This is likely due to flights never departing due to cancellation. This should not be too much of an issue as we still have over 30 million rows to work with and we avoid the issue of data leakage as we want our weather measurements to be recorded at least 2 hours before a flight's scheduled or actual departure, whichever comes earlier.

In [0]:
# Ensure no duplicate columns
import collections
duplicate_cols = [item for item, count in collections.Counter(otpw_full_2h_6h.columns).items() if count > 1]
print(f"Duplicate columns: {duplicate_cols}")

# Split into train and test sets before conducting feature engineering to prevent data leakage. Also include functions to split and rejoin train data to respect rolling year-over-year CV in modeling pipeline

In [0]:
def otpw_train_test_split(otpw_df):
    '''Split otpw data into train (2015-2018) and blind test (2019) sets'''

    train_df = otpw_df.filter(col("YEAR").cast("int") <= 2018)
    test_df = otpw_df.filter(col("YEAR").cast("int") >= 2019)

    return train_df, test_df

def split_otpw_train_data(otpw_train_df):
    '''Splits train data by year.'''
    df_2015 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2015) &
        (col("YEAR").cast("int") <= 2015)
    )
    df_2016 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2016) &
        (col("YEAR").cast("int") <= 2016)
    )
    df_2017 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2017) &
        (col("YEAR").cast("int") <= 2017)
    )
    df_2018 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2018) &
        (col("YEAR").cast("int") <= 2018)
    )
    return df_2015, df_2016, df_2017, df_2018

def rejoin_otpw_train_splits(df_2015, df_2016, df_2017, df_2018):
    '''Rejoins the train data by year back into a single dataset'''
    return reduce(DataFrame.union, [df_2015, df_2016, df_2017, df_2018])
    
otpw_train_df, otpw_test_df = otpw_train_test_split(otpw_full_2h_6h)

# Extract hour from scheduled departure, cast DISTANCE to double

In [0]:
# Add DEP_HOUR column from sched_depart_date_time
otpw_train_df = otpw_train_df.withColumn("DEP_HOUR", hour("sched_depart_date_time"))
otpw_test_df = otpw_test_df.withColumn("DEP_HOUR", hour('sched_depart_date_time'))

# Cast DISTANCE to double
otpw_train_df = otpw_train_df.withColumn("DISTANCE", otpw_train_df["DISTANCE"].cast("double"))
otpw_test_df = otpw_test_df.withColumn("DISTANCE", otpw_test_df["DISTANCE"].cast("double"))

# Verify the min and max dates for each split

In [0]:
%skip
otpw_train_df.select(
    min("sched_depart_date_time").alias('min_date'),
    max("sched_depart_date_time").alias('max_date')
).show()

otpw_test_df.select(
    min("sched_depart_date_time").alias('min_date'),
    max("sched_depart_date_time").alias('max_date')
).show()

# Correlation (on the training set from 2015-2018)

In [0]:
columns = [
    "DEP_DEL15",
    "DISTANCE",
    "CRS_ELAPSED_TIME",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "2h_HourlyDryBulbTemperature",
    "2h_HourlyDewPointTemperature",
    "2h_HourlyWindSpeed",
    "2h_HourlyWindGustSpeed",
    "2h_HourlyWindDirection",
    "2h_HourlyVisibility",
    "2h_HourlyPrecipitation",
    "2h_HourlyRelativeHumidity",
    "2h_HourlyStationPressure",
    "2h_HourlySeaLevelPressure"
]

# df_casted = df.select([col(c).cast("double").alias(c) for c in columns])
otpw_train_df_casted = otpw_train_df.select([col(c).cast("double").alias(c) for c in columns])


In [0]:
from pyspark.sql.functions import when

otpw_train_df_cleaned = otpw_train_df.select([
    when(col(c).isin("", "NA", "null"), None)
    .otherwise(col(c))
    .cast("double")
    .alias(c)
    for c in columns
]).dropna()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(inputCols=columns, outputCol="features")
df_vector = assembler.transform(otpw_train_df_cleaned)

corr_matrix = Correlation.corr(df_vector, "features", method="spearman").head()[0]

corr_matrix.toArray()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

corr_array = corr_matrix.toArray()
corr_df = pd.DataFrame(corr_array, index=columns, columns=columns)

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5
)
plt.title("Spearman Correlation Heatmap")
plt.show()

# General Info and Feature Types

In [0]:
# Dataset dimensions
num_rows = otpw_train_df.count()
num_cols = len(otpw_train_df.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

# Handle Duplicates and Missing Data

## Duplicates

In [0]:
total = otpw_train_df.count()
distinct = otpw_train_df.dropDuplicates().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicate rows: {total - distinct}")

## Missing Data for Response Variable

In [0]:
delayed = otpw_train_df.filter(col("DEP_DEL15") == 1)
on_time = otpw_train_df.filter(col("DEP_DEL15") == 0)
missing = otpw_train_df.filter(col("DEP_DEL15").isNull())

print(f"Total delayed flights: {delayed.count()}")
print(f"Total on-time flights: {on_time.count()}")
print(f"Missing DEP_DEL15: {missing.count()}")
print(f"Distinct flights: {distinct}")
print(f"Missing flight percentage (%): {missing.count() / distinct * 100:.2f}")

We can see that DEP_DEL15, the variable we are interested in, has 0.02 percent missing data. This could be due to the fact that there are cancelled flights that may have null DEP_DEL15 values.

In [0]:
# Is it true that all cancelled flights will not have DEP_DELAY and DEP_DEL15?
cancelled = otpw_train_df.filter(col("CANCELLED") == 1)
total_cancelled = cancelled.count()
cancelled_with_delay = cancelled.filter(col('DEP_DELAY').isNotNull()).count()
cancelled_with_del15 = cancelled.filter(col('DEP_DEL15').isNotNull()).count()

print(f"Total cancelled flights: {total_cancelled} ({total_cancelled / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DELAY: {cancelled_with_delay} ({cancelled_with_delay / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DEL15: {cancelled_with_del15} ({cancelled_with_del15 / num_rows * 100:.2f}% of dataset)")

We decide to remove cancelled flights as they do not contribute towards flight delay predictions on an already unbalanced dataset.

In [0]:
# Filter out cancelled flights
otpw_train_df = otpw_train_df.filter(col("CANCELLED") == 0)
num_rows = otpw_train_df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
missing = otpw_train_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing.count()}")

There are still 4744 flights with no departure delay label. We can impute these missing values by computing the difference between CRS_DEP_TIME and DEP_TIME.

In [0]:
def impute_missing_target_var(otpw_df):
    # 1. Build full timestamps for scheduled and actual departure times
    otpw_filled = otpw_df.withColumn(
        "sched_ts",
        to_timestamp(
            concat(
                col("FL_DATE"), 
                lpad(col("CRS_DEP_TIME"), 4, "0")
            ),
            "yyyy-MM-ddHHmm"
        )
    ).withColumn(
        "actual_ts",
        to_timestamp(
            concat(
                col("FL_DATE"), 
                lpad(col("DEP_TIME"), 4, "0")
            ),
            "yyyy-MM-ddHHmm"
        )
    )

    # 2. Compute difference in minutes
    otpw_filled = otpw_filled.withColumn(
        "dep_diff_mins",
        (col("actual_ts").cast("long") - col("sched_ts").cast("long")) / 60
    )

    # 3. Fill missing DEP_DEL15 using the computed difference
    otpw_filled = otpw_filled.withColumn(
        "DEP_DEL15",
        when(
            col("DEP_DEL15").isNull(),
            when(col("dep_diff_mins") >= 15, 1).otherwise(0)
        ).otherwise(col("DEP_DEL15")).cast("double")
    )

    return otpw_filled

In [0]:
otpw_train_df = impute_missing_target_var(otpw_train_df)
missing_dep_del15_train = otpw_train_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing_dep_del15_train.count()}")

We will also removed cancelled flights from the test set and impute the missing target variable "DEP_DEL15" as well. This will not introduce data leakage as cancelled flights and the imputation is independent from all other flights. That is, the flight's overall status and departure depends on the flight's own metadata.

In [0]:
otpw_test_df = otpw_test_df.filter(col("CANCELLED") == 0)
otpw_test_df = impute_missing_target_var(otpw_test_df)
missing_dep_del15_test = otpw_test_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing_dep_del15_test.count()}")

There is no more missing data for `DEP_DEL15` in the train and test sets.

In [0]:
# Class balance of DEP_DEL15
target_dist = otpw_train_df.groupBy("DEP_DEL15").count().toPandas()
target_dist["percentage"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(2)
print(target_dist)

plt.figure(figsize=(6, 4))
plt.bar(target_dist["DEP_DEL15"].astype(str), target_dist["count"])
plt.title("Flight Delay Distribution (DEP_DEL15)")
plt.xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
plt.ylabel("Count")
for i, row in target_dist.iterrows():
    plt.text(i, row["count"], f"{row['percentage']}%", ha="center", va="bottom")
plt.tight_layout()
# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')
plt.show()

%md
# Tail number delay propagation
For each flight, look up the same aircraft's previous leg using `TAIL_NUM` and check if that prior flight was delayed. That's the propagation effect that's going to be very powerful.

Create two features: 

1. `prev_leg_delayed` (binary: was the previous flight on this aircraft delayed?) 

2. `prev_leg_delay_minutes` (how many minutes was it delayed?). This captures the cascading effect: if an aircraft arrived late from its previous route, it's likely to depart late on its next one. 

We could sort by `TAIL_NUM` and `sched_depart_date_time`, then use a lag window to get the prior flight's `DEP_DEL15` and `DEP_DELAY`. Just note, we should be careful about the leakage. 

In [0]:
# ----------------------------
# 1. FL_DATE as date
# ----------------------------
otpw_train_df = otpw_train_df.withColumn("FL_DATE_TS", F.to_date("FL_DATE"))
otpw_test_df = otpw_test_df.withColumn("FL_DATE_TS", F.to_date("FL_DATE"))

# ----------------------------
# 2. Convert HHMM → minutes since midnight
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "dep_min",
    (F.col("CRS_DEP_TIME") / 100).cast("int") * 60 + (F.col("CRS_DEP_TIME") % 100)
).withColumn(
    "arr_min",
    (F.col("CRS_ARR_TIME") / 100).cast("int") * 60 + (F.col("CRS_ARR_TIME") % 100)
)

otpw_test_df = otpw_test_df.withColumn(
    "dep_min",
    (F.col("CRS_DEP_TIME") / 100).cast("int") * 60 + (F.col("CRS_DEP_TIME") % 100)
).withColumn(
    "arr_min",
    (F.col("CRS_ARR_TIME") / 100).cast("int") * 60 + (F.col("CRS_ARR_TIME") % 100)
)

# ----------------------------
# 3. Detect overnight flight
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "day_offset",
    F.when(F.col("arr_min") < F.col("dep_min"), 1).otherwise(0)
)

otpw_test_df = otpw_test_df.withColumn(
    "day_offset",
    F.when(F.col("arr_min") < F.col("dep_min"), 1).otherwise(0)
)

# ----------------------------
# 4. Convert everything to seconds since epoch
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "base_ts",
    F.unix_timestamp("FL_DATE_TS")
).withColumn(
    "crs_dep_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("dep_min") * 60)
).withColumn(
    "crs_arr_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("arr_min") * 60 + F.col("day_offset") * 86400)
)

otpw_test_df = otpw_test_df.withColumn(
    "base_ts",
    F.unix_timestamp("FL_DATE_TS")
).withColumn(
    "crs_dep_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("dep_min") * 60)
).withColumn(
    "crs_arr_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("arr_min") * 60 + F.col("day_offset") * 86400)
)

In [0]:
window_tail = Window.partitionBy("TAIL_NUM") \
                    .orderBy("sched_depart_date_time")

# get the previous delay time and if the flight was delayed
otpw_train_df = otpw_train_df.withColumn(
    "prev_DEP_DEL15",
    F.lag("DEP_DEL15", 1).over(window_tail)
).withColumn(
    "prev_DEP_DELAY",
    F.lag("DEP_DELAY", 1).over(window_tail)
)

otpw_test_df = otpw_test_df.withColumn(
    "prev_DEP_DEL15",
    F.lag("DEP_DEL15", 1).over(window_tail)
).withColumn(
    "prev_DEP_DELAY",
    F.lag("DEP_DELAY", 1).over(window_tail)
)

In [0]:
# turnaround time

# ----------------------------------------
# NEW: previous arrival time
# ----------------------------------------
otpw_train_df = otpw_train_df.withColumn(
    "prev_arrival_ts",
    F.lag("crs_arr_datetime", 1).over(window_tail)
)

otpw_test_df = otpw_test_df.withColumn(
    "prev_arrival_ts",
    F.lag("crs_arr_datetime", 1).over(window_tail)
)


# ----------------------------------------
# NEW: time between flights (turnaround)
# ----------------------------------------
otpw_train_df = otpw_train_df.withColumn(
    "turnaround_minutes",
    (F.col("sched_depart_date_time").cast("long") -
     F.col("prev_arrival_ts").cast("long")) / 60
)

otpw_test_df = otpw_test_df.withColumn(
    "turnaround_minutes",
    (F.col("sched_depart_date_time").cast("long") -
     F.col("prev_arrival_ts").cast("long")) / 60
)

## Other missing data explanation

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

We first check report type for missing weather data. One of the reasons why there are so many missing values is that the weather dataset has multiple report types (FM-15, FM-16, FM-12, etc) that do not contain all information. For example:

* FM-15: Standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* FM-16: Special weather report issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* FM-12: Synoptic report issued every 6–12 hours with fewer variables; many fields are missing and it does not align well with hourly flight data.
* SOD (Summary of Day): Daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* SHEF and SY-MT: Rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

Because of this, we focus primarily on FM-15 for modeling. In the future we can optionally include FM-16 if we want to capture extreme weather events. As for the other report types, we will drop them to ensure consistent and reliable features for each flight.

In [0]:
%skip
otpw_train_df.groupBy("REPORT_TYPE") \
  .agg(count("*").alias("count")) \
  .display()

In [0]:
otpw_train_df = otpw_train_df.filter(col("REPORT_TYPE") == "FM-15")
num_rows = otpw_train_df.count()
print(f"After filtering FM-15: {num_rows:,} rows remaining")

Next, we look into source type, since the weather dataset is based off of many different sources, there may be sources that have more missing data than others.

In [0]:
%skip
otpw_train_df.groupBy("SOURCE").count().display()

In [0]:
%skip
weather_cols = [
    "2h_HourlyAltimeterSetting",
    "2h_HourlyDewPointTemperature",
    "2h_HourlyDryBulbTemperature",
    "2h_HourlyPrecipitation",
    "2h_HourlyPresentWeatherType",
    "2h_HourlyPressureChange",
    "2h_HourlyPressureTendency",
    "2h_HourlyRelativeHumidity",
    "2h_HourlySkyConditions",
    "2h_HourlySeaLevelPressure",
    "2h_HourlyStationPressure",
    "2h_HourlyVisibility",
    "2h_HourlyWetBulbTemperature",
    "2h_HourlyWindDirection",
    "2h_HourlyWindGustSpeed",
    "2h_HourlyWindSpeed"
]

exprs = [
    (count(when(col(c).isNull(), True)) / count("*")).alias(f"{c}_missing_pct")
    for c in weather_cols
]

otpw_train_df.groupBy("SOURCE").agg(*exprs).display()

In the table, we can see that source 7 and 6 make up the majority of the dataset. There are missing data, but there are also a lot of columns that have are filled out in the majority of rows.

Additionally based off of the data dictionary to the original data, null values represent zero or not observed, and M represents missing data. 

In the OTPW dataset, there are both 0 and null values in precipitation, but we believe that is just an artifact of merging the data. Specifically when the report type is FM-15 (regular at hourly intervals), we should treat nulls as 0s.

In [0]:
from pyspark.sql.types import FloatType

weather_continuous_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

# Cast continuous cols to float
for c in weather_continuous_cols:
    otpw_train_df = otpw_train_df.withColumn(f"2h_{c}", col(f"2h_{c}").cast(FloatType()))
    otpw_train_df = otpw_train_df.withColumn(f"6h_{c}", col(f"6h_{c}").cast(FloatType()))
    otpw_test_df = otpw_test_df.withColumn(f"2h_{c}", col(f"2h_{c}").cast(FloatType()))
    otpw_test_df = otpw_test_df.withColumn(f"6h_{c}", col(f"6h_{c}").cast(FloatType()))
    if f"2h_{c}" in {"2h_HourlyPrecipitation"}:
        otpw_train_df = otpw_train_df.na.fill({f"2h_{c}": 0})  # fill nulls column by column
        otpw_test_df = otpw_test_df.na.fill({f"2h_{c}": 0})
    if f"6h_{c}" in {"6h_HourlyPrecipitation"}:
        otpw_train_df = otpw_train_df.na.fill({f"6h_{c}": 0})  # fill nulls column by column
        otpw_test_df = otpw_test_df.na.fill({f"6h_{c}": 0})

# Show percentage of null values for weather columns

In [0]:
def get_null_info(df):
    '''Display number of null records and null percentage for each column in the dataframe'''
    total_rows = df.count()

    exprs = [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    counts = df.select(exprs)

    # Convert wide → long
    long_df = counts.select(
        F.explode(F.map_from_arrays(
            F.array(*[F.lit(c) for c in df.columns]),
            F.array(*[F.col(c) for c in df.columns])
        )).alias("column", "null_count")
    )

    # Add percentage
    return long_df.withColumn(
        "null_pct", 
        F.concat(
            F.round((F.col("null_count") / total_rows)*100, 2).cast("string"),
            F.lit("%")
        )
    )

In [0]:
otpw_train_df_weather_null_info = get_null_info(otpw_train_df.select(weather_continuous_cols))
display(otpw_train_df_weather_null_info)

### Compute the weather trend features (subtract observations 6 hrs before departure with observations 2 hrs before)

In [0]:
# Did not include precipitation because most values are close to 0.
# If we include Visibility trend we need to remove visiblityCat to avoid multicollinearity
weather_features = [
  "DryBulbTemperature",
  "WindSpeed",
  "RelativeHumidity",
  "Visibility"
]

for feature in weather_features:
  otpw_train_df = otpw_train_df.withColumn(
    f"{feature}_trend_2h_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  otpw_test_df = otpw_test_df.withColumn(
    f"{feature}_trend_2h_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  print(f"{feature}_trend_2h_6h")

# Standardization and Transformations of Weather Trend Features

Based off of the table above showing the missingness of our weather data, we drop `HourlyWindGustSpeed`, `HourlyPressureChange`, `HourlyPressureTendency` due to severe missingness (>50%). We also remove, `HourlyAltimeterSetting`, `HourlySeaLevelPressure`, and `HourlyStationPressure` because they are unlikely to provide any predictive value toward delays. Since `HourlyWetBulbTemperature` and `HourlyDewPointTemperature` are highly correlated with `HourlyDryBulbTemperature` (which is primarily used as the standard air temperature
reported), we keep only `HourlyDryBulbTemperature`. Finally, because `HourlyWindDirection` is correlated with `HourlyWindSpeed`, the latter of which might be more relevant towards flight delays, we drop `HourlyWindDirection`.

This leaves us with `HourlyPrecipitation`, `HourlyRelativeHumidity`, `HourlyWindSpeed`, `HourlyVisibility` and `HourlyDryBulbTemperature` as our flight delay signals. We will use them in our modeling, and they will need to be standardized and transformed.

In [0]:
# Collect data to pandas (small enough sample!)
precip_pd = otpw_train_df.select("2h_HourlyPrecipitation").sample(0.1).toPandas()  # sample 10% to speed up

plt.figure(figsize=(8,5))
plt.hist(precip_pd['2h_HourlyPrecipitation'], bins=50)
plt.xlabel('Hourly Precipitation')
plt.ylabel('Count')
plt.title('Distribution of Hourly Precipitation')

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

In [0]:
import numpy as np
from pyspark.sql.functions import log1p

plt.hist(precip_pd['2h_HourlyPrecipitation'].apply(lambda x: np.log1p(x)), bins=50)
plt.xlabel('Log(Precipitation + 1)')
plt.ylabel('Count')
plt.title('Log-transformed Hourly Precipitation')

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

Precipitation is heavily skewed, even after log transformations it is still skewed. A better option would be to turn it into a binary feature where 0 = no precipitation and 1 = precipitation

In [0]:
otpw_train_df = otpw_train_df.withColumn(
    "PrecipBinary",
    when(col("2h_HourlyPrecipitation") > 0, 1).otherwise(0)
)

otpw_test_df = otpw_test_df.withColumn(
    "PrecipBinary",
    when(col("2h_HourlyPrecipitation") > 0, 1).otherwise(0)
)

In [0]:
# Sample data to pandas (optional for large datasets)
sample_pd = otpw_train_df.select("PrecipBinary").toPandas()

# Count occurrences of 0 and 1
counts = sample_pd["PrecipBinary"].value_counts().sort_index()

# Compute percentages
percentages = counts / counts.sum() * 100

# Plot bar chart
plt.figure(figsize=(6,4))
bars = plt.bar(counts.index.astype(str), counts.values)

plt.xticks([0,1])
plt.xlabel("Precipitation Binary (0 = No Rain, 1 = Any Rain)")
plt.ylabel("Count")
plt.title("Distribution of Binary Precipitation")

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

# Add percentage labels on top of bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    pct = percentages.iloc[i]
    ct = counts.iloc[i]
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f"{ct} ({pct:.1f}%)", ha='center', va='bottom')

plt.show()

Next we look at the other numeric features: humidity, wind speed, visibility and temperature.

In [0]:
weather_numeric_cols = [
    "RelativeHumidity_trend_2h_6h",
    "WindSpeed_trend_2h_6h",
    "Visibility_trend_2h_6h",
    "DryBulbTemperature_trend_2h_6h"
]

sample_pd = otpw_train_df.select(weather_numeric_cols).sample(0.1, seed=42).toPandas()

plt.figure(figsize=(16,10))

for i, col_name in enumerate(weather_numeric_cols):
    ax = plt.subplot(2, 2, i+1)

    if col_name == "WindSpeed_trend_2h_6h":
        ax.set_xlim([-50, 50])
        ax.hist(sample_pd[col_name], bins=1000)
    else:
        ax.hist(sample_pd[col_name], bins=30)
    ax.set_title(f'Distribution of {col_name}')
    ax.set_xlabel(col_name)
    ax.set_ylabel('Count')

    

    # Disable scientific notation for THIS subplot
    ax.ticklabel_format(style='plain', axis='y')


plt.tight_layout()
plt.show()


All of these weather trends have varying magnitudes of scale. For example, relative humidity trend has a scale of -100 to 100, while DryBulbTemperature trend only goes from -40 to 40.

To prevent large-magnitude features from dominating models, we will standardize these features. This will help our logistic regression baseline converge.

Because we are performing rolling cross-validation year-over-year, we will split the training data by year so we have a total of 4 dataframes (2015, 2016, 2017, 2018). This way we will be computing statistics on each individual year for standardization to mitigate any data leakage.

In [0]:
otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

# Standardize relative humidity and dry bulb temperature

In [0]:
def standardize_features(otpw_data, features = [], stats = None):
    # Compute stats in one aggregation
    agg_exprs = [mean(c).alias(f"{c}_mean") for c in features] + \
                [stddev(c).alias(f"{c}_std") for c in features]

    if stats is None:
        stats = otpw_data.agg(*agg_exprs).first()

    # Apply standardization to each feature using the current fold's statistics
    for c in features:
        mu = stats[f"{c}_mean"]
        sd = stats[f"{c}_std"]
        otpw_data = otpw_data.withColumn(
            f"{c}_stdized", 
            (col(c) - mu) / sd
        )
    return otpw_data

def get_mean_and_std(otpw_data, features):
    # Compute stats in one aggregation
    agg_exprs = [mean(c).alias(f"{c}_mean") for c in features] + \
                [stddev(c).alias(f"{c}_std") for c in features]

    stats = otpw_data.agg(*agg_exprs).first()

    return stats

# Standardize on each fold, then join folds back together

In [0]:
weather_trend_features = [
  "DryBulbTemperature_trend_2h_6h",
  "WindSpeed_trend_2h_6h",
  "RelativeHumidity_trend_2h_6h",
  "Visibility_trend_2h_6h"
]

otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = standardize_features(otpw_2015_df, weather_trend_features)
otpw_2016_df = standardize_features(otpw_2016_df, weather_trend_features)
otpw_2017_df = standardize_features(otpw_2017_df, weather_trend_features)
otpw_2018_df = standardize_features(otpw_2018_df, weather_trend_features)

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)

# Get entire train stats to use for standardizing test set
otpw_train_stats = get_mean_and_std(otpw_train_df, weather_trend_features)
otpw_test_df = standardize_features(otpw_test_df, weather_trend_features, stats = otpw_train_stats)

# Plot standardized distribution of humidity and temperature

In [0]:
# Columns to plot
stdized_cols = ["DryBulbTemperature_trend_2h_6h_stdized", "RelativeHumidity_trend_2h_6h_stdized", 'Visibility_trend_2h_6h_stdized', "WindSpeed_trend_2h_6h_stdized",]

sample_pd = otpw_train_df.select(stdized_cols).toPandas()

# Plot histograms
plt.figure(figsize=(15, 10))

for i, col_name in enumerate(stdized_cols):
    ax = plt.subplot(2, 2, i+1)
    ax.hist(sample_pd[col_name], bins=50)
    ax.set_title(f'Distribution of {col_name}')
    ax.set_xlabel(col_name)
    ax.set_ylabel('Count')

    # Disable scientific notation for THIS subplot
    ax.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

# Log Transform and standardize Wind Speed

In [0]:
%skip
# Log-transform (creates a new column)
otpw_train_df = otpw_train_df.withColumn("WindSpeed_trend_2h_6h_log", log1p(col("WindSpeed_trend_2h_6h")))

# Split train data by year again before standardizing features
otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = standardize_features(otpw_2015_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2016_df = standardize_features(otpw_2016_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2017_df = standardize_features(otpw_2017_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2018_df = standardize_features(otpw_2018_df, features = ["WindSpeed_trend_2h_6h_log"])

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)
otpw_train_stats = get_mean_and_std(otpw_train_df, features = ["WindSpeed_trend_2h_6h_log"])

otpw_test_df = otpw_test_df.withColumn("WindSpeed_trend_2h_6h_log", log1p(col("WindSpeed_trend_2h_6h")))
otpw_test_df = standardize_features(otpw_test_df, features = ['WindSpeed_trend_2h_6h_log'], stats = otpw_train_stats)

In [0]:
%skip
# Log-transform (creates a new column)
otpw_train_df = otpw_train_df.withColumn("2h_HourlyWindSpeed_log", log1p(col("2h_HourlyWindSpeed")))

# Split train data by year again before standardizing features
otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

In [0]:
%skip
otpw_2015_df = standardize_features(otpw_2015_df, features = ['2h_HourlyWindSpeed_log'])
otpw_2016_df = standardize_features(otpw_2016_df, features = ['2h_HourlyWindSpeed_log'])
otpw_2017_df = standardize_features(otpw_2017_df, features = ['2h_HourlyWindSpeed_log'])
otpw_2018_df = standardize_features(otpw_2018_df, features = ['2h_HourlyWindSpeed_log'])

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)
otpw_train_stats = get_mean_and_std(otpw_train_df, features = ['2h_HourlyWindSpeed_log'])

otpw_test_df = otpw_test_df.withColumn("2h_HourlyWindSpeed_log", log1p(col("2h_HourlyWindSpeed")))
otpw_test_df = standardize_features(otpw_test_df, features = ['2h_HourlyWindSpeed_log'], stats = otpw_train_stats)

In [0]:
%skip
sample_pd = otpw_train_df.select("WindSpeed_trend_2h_6h_log_stdized").toPandas()

# Plot histogram
plt.figure(figsize=(8,5))
plt.hist(sample_pd["WindSpeed_trend_2h_6h_log_stdized"], bins=50)
plt.xlabel("WindSpeed_trend_2h_6h_log_stdized")
plt.ylabel("Count")
plt.title("Histogram of Standardized Log-Transformed Wind Speed trend")

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()


# Transform Visibility into a categorical variable with the following categories:
| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

In [0]:
%skip
from pyspark.sql.functions import when, col

# Categorize visibility on train set
otpw_train_df = otpw_train_df.withColumn(
    "VisibilityCat",
    when(col("2h_HourlyVisibility") < 2, 0)
    .when((col("2h_HourlyVisibility") >= 2) & (col("2h_HourlyVisibility") < 5), 1)
    .when((col("2h_HourlyVisibility") >= 5) & (col("2h_HourlyVisibility") < 10), 2)
    .otherwise(3)
)

# Categorize visibility on test set. No leakage because test set does not influence learned parameters or use future info.
otpw_test_df = otpw_test_df.withColumn(
    "VisibilityCat",
    when(col("2h_HourlyVisibility") < 2, 0)
    .when((col("2h_HourlyVisibility") >= 2) & (col("2h_HourlyVisibility") < 5), 1)
    .when((col("2h_HourlyVisibility") >= 5) & (col("2h_HourlyVisibility") < 10), 2)
    .otherwise(3)
)

In [0]:
%skip
# Sample to Pandas (optional for large datasets)
sample_pd = otpw_train_df.select("VisibilityCat").sample(0.1, seed=42).toPandas()

# Count occurrences of each category
counts = sample_pd["VisibilityCat"].value_counts().sort_index()

# Plot bar chart
plt.figure(figsize=(6,4))
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Visibility Category (0=Very Low, 3=High)")
plt.ylabel("Count")
plt.title("Distribution of Visibility Categories")
plt.xticks(counts.index)

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

In [0]:
%skip
# Sample to Pandas (optional for large datasets)
sample_pd = otpw_train_df.select("VisibilityCat").toPandas()

# Count occurrences of each category
counts = sample_pd["VisibilityCat"].value_counts().sort_index()

# Compute percentages
percentages = counts / counts.sum() * 100

# Plot bar chart with counts on y-axis
plt.figure(figsize=(6,4))
bars = plt.bar(counts.index.astype(str), counts.values)

plt.xlabel("Visibility Category (0=Very Low, 1=Low, 2=Moderate, 3=High)")
plt.ylabel("Count")
plt.title("Distribution of Visibility Categories")
plt.xticks(counts.index)

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

# Add percentage labels on top of bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    pct = percentages.iloc[i]
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f"{pct:.1f}%", ha='center', va='bottom')

plt.show()

# Additional Feature Engineering and Creation

We will create two additional features, measuring if the flight is on a weekend, as well as and combination of the origin and destination airport.

In [0]:
otpw_train_df = otpw_train_df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

In [0]:
otpw_train_df.groupBy("IsWeekend").count().display()

In [0]:
from pyspark.sql.functions import concat_ws, col

otpw_train_df = otpw_train_df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)

In [0]:
# Perform the same weekend and route feature creations on the test set

otpw_test_df = otpw_test_df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

otpw_test_df = otpw_test_df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)


We will need to encode this new variable for ML models

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Step 1: Convert Route string into numeric index
route_indexer = StringIndexer(inputCol="Route", outputCol="RouteIndex", handleInvalid="keep")
# otpw_train_df = route_indexer.fit(otpw_train_df).transform(otpw_train_df)
route_indexer_model = route_indexer.fit(otpw_train_df)
otpw_train_df = route_indexer_model.transform(otpw_train_df)

# Step 2: One-hot encode the index
route_encoder = OneHotEncoder(inputCols=["RouteIndex"], outputCols=["RouteVec"])
# otpw_train_df = route_encoder.fit(otpw_train_df).transform(otpw_train_df)
route_encoder_model = route_encoder.fit(otpw_train_df)
otpw_train_df = route_encoder_model.transform(otpw_train_df)

In [0]:
# Perform the same one-hot encoding on test set using training-fitted indexer and encoder
otpw_test_df = route_indexer_model.transform(otpw_test_df)
otpw_test_df = route_encoder_model.transform(otpw_test_df)

# Write train and test sets for OTPW to TEAM_DIR

In [0]:
# otpw_train_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041426_otpw_train_df.parquet")
# otpw_test_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041426_otpw_test_df.parquet")

# Read in checkpointed train and test data.

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041426_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041426_otpw_test_df.parquet")

# Daniel: We create a window 2-4 hours before the flight to see the count percentange of delayed flights. 

If there are no flights in that window, it is it counts as 0 (nulls are replaced with 0). We can change the flight window to capture a longer time period before the flight's departure. The windows are grouped by airport (not enough data for it to be grouped by airline in my opinion).

In [0]:
def compute_window_features(otpw_df, lookback_start_hours = -4, lookback_end_hours = -2):
    '''Compute two window-based features from OTPW dataset:
    
        1. The number of delayed flights partitioned by airport in the specified lookback window.

        2. The percentange of delayed flights partitioned by airport in the specified lookback window.

    Default lookback window is 2-4 hours before departure. If there are no flights in the window, we count the # of flights as 0.
    '''
    # Repartition for performance
    otpw_df = otpw_df.repartition("ORIGIN")

    # Define time-based window containing flights based on specified lookback hours before departure
    window_spec = Window.partitionBy("ORIGIN") \
        .orderBy(F.col("sched_depart_date_time").cast("long")) \
        .rangeBetween(lookback_start_hours * 3600, lookback_end_hours * 3600) # can change these number for different window

    # The number of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"delayed_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.sum("DEP_DEL15").over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    # The total number of flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"total_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.count("*").over(window_spec),
            F.lit(0)
        )
    )

    # The percentange of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"delay_pct_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.avg(F.coalesce(F.col("DEP_DEL15"), F.lit(0))).over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    return otpw_df

In [0]:
otpw_train_df = compute_window_features(otpw_train_df)
otpw_test_df = compute_window_features(otpw_test_df)

We create two additional features above:

1. `delayed_flights_2_4hr_before` gives the number of delayed flights for each airport in the 2-4 hours before this flight.

2. `delay_pct_flights_2_4hr_before` is the percentage of delayed flights for each airport in the same window.

Both features capture recently operational conditions at the airport which we hope carry a predictive signal towards future delays.

However, including delay flight percentage and delayed flight count in logistic regression can result in multicollinearity issues. To avoid multicollinearity for logistic regression, we will use `delay_pct_flights_2_4hr_before` as a normalized measure of congestion, include and log-transform a new feature representing total flight volume `total_flights_2_4hr_before`, and drop `delayed_flights_2_4hr_before`.

In [0]:
# Log transform total flights partitioned by airport
otpw_train_df = otpw_train_df.withColumn(
    "log_total_flights_2_4hr_before",
    log1p(col("total_flights_2_4hr_before"))
)

otpw_test_df = otpw_test_df.withColumn(
    "log_total_flights_2_4hr_before",
    log1p(col("total_flights_2_4hr_before"))
)

# Create another window looking 7 days back to 24 hours

In [0]:
otpw_train_df = compute_window_features(otpw_train_df, lookback_start_hours = -24*7, lookback_end_hours = -24)
otpw_test_df = compute_window_features(otpw_test_df, lookback_start_hours = -24*7, lookback_end_hours = -24)

In [0]:
assert 'total_flights_24_168hr_before' in otpw_train_df.columns
assert 'total_flights_24_168hr_before' in otpw_test_df.columns

In [0]:
#  Log transform total flight volume for 1 week window
otpw_train_df = otpw_train_df.withColumn(
    "log_total_flights_24_168hr_before",
    log1p(col("total_flights_24_168hr_before"))
)

otpw_test_df = otpw_test_df.withColumn(
    "log_total_flights_24_168hr_before",
    log1p(col("total_flights_24_168hr_before"))
)

# Create Graph features - Unweighted and Weighted PageRank

In [0]:
def compute_unweighted_pagerank_features(otpw_df):
    """
    Compute unweighted PageRank-based graph features.

    Unweighted PageRank: Every edge (Origin -> Destination) counts equally, and measures pure network centrality.
    """

    # Build edges for OTPW_DF
    edges = (
        otpw_df
        .filter(col("ORIGIN").isNotNull() & col("DEST").isNotNull())
        .groupBy("ORIGIN", "DEST")
        .agg(F.count("*").alias("FLIGHT_COUNT"))
        .withColumnRenamed("ORIGIN", "src")
        .withColumnRenamed("DEST", "dst")
        .withColumnRenamed('FLIGHT_COUNT', 'WEIGHT')
    )

    vertices = (
        edges.select(col('src').alias("id"))
        .union(edges.select(col('dst').alias("id")))
        .distinct()
    )

    # Build unweighted graph
    unweighted_graph = GraphFrame(vertices, edges.select("src", "dst"))

    # Compute PageRank
    pagerank_unweighted_graph = unweighted_graph.pageRank(
        resetProbability=0.15,
        maxIter=20
    ).vertices.select(
        col("id").alias("AIRPORT"),
        col("pagerank").alias("unweighted_pagerank")
    )

    # Join ORIGIN
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_unweighted_graph.withColumnRenamed("AIRPORT", "ORIGIN")), 
            on="ORIGIN", 
            how="left"
        )
        .withColumnRenamed("unweighted_pagerank", "origin_pagerank_unweighted")
    )

    # Join DEST
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_unweighted_graph.withColumnRenamed("AIRPORT", "DEST")), 
            on="DEST", 
            how="left"
        )
        .withColumnRenamed("unweighted_pagerank", "dest_pagerank_unweighted")
    )

    return otpw_df
    

def compute_weighted_pagerank_features(otpw_df):
    """
    Compute weighted PageRank-based graph features.

    Weighted PageRank: Edges are weighted by the # of flights between airports. Airports with heavy traffic influence the graph more.
    """
    # Build edges for OTPW_DF
    edges = (
        otpw_df
        .filter(col("ORIGIN").isNotNull() & col("DEST").isNotNull())
        .groupBy("ORIGIN", "DEST")
        .agg(F.count("*").alias("FLIGHT_COUNT"))
        .withColumnRenamed("ORIGIN", "src")
        .withColumnRenamed("DEST", "dst")
        .withColumnRenamed('FLIGHT_COUNT', 'WEIGHT')
    )

    vertices = (
        edges.select(col('src').alias("id"))
        .union(edges.select(col('dst').alias("id")))
        .distinct()
    )

    # Duplicate each edge according to its weight (i.e. weight = 500 means 500 copies of the edge between src-dst.)
    window = Window.partitionBy("src")
    edges_weighted = (
        edges
        .withColumn("seq", F.sequence(F.lit(0), col("weight").cast('int') - 1))
        .select("src", "dst", F.explode("seq"))
        .select("src", "dst")
    )


    # Build weighted graph
    weighted_graph = GraphFrame(vertices, edges_weighted)

    # Compute weighted PageRank
    pr = weighted_graph.pageRank(
        resetProbability=0.15,
        maxIter=20,
    )

    pagerank_weighted_graph = pr.vertices.select(
        col("id").alias("AIRPORT"),
        col("pagerank").alias("weighted_pagerank")
    )

    # Join ORIGIN
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_weighted_graph.withColumnRenamed("AIRPORT", "ORIGIN")), 
            on="ORIGIN", 
            how="left")
        .withColumnRenamed("weighted_pagerank", "origin_pagerank_weighted")
    )

    # Join DEST
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_weighted_graph.withColumnRenamed("AIRPORT", "DEST")), 
            on="DEST", 
            how="left")
        .withColumnRenamed("weighted_pagerank", "dest_pagerank_weighted")
    )

    return otpw_df

# Compute weighted and unweighted pagerank features for train and test sets

In [0]:
# otpw_train_df = compute_weighted_pagerank_features(otpw_train_df)
# otpw_train_df = compute_unweighted_pagerank_features(otpw_train_df)
otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = compute_unweighted_pagerank_features(compute_weighted_pagerank_features(otpw_2015_df))
otpw_2016_df = compute_unweighted_pagerank_features(compute_weighted_pagerank_features(otpw_2016_df))
otpw_2017_df = compute_unweighted_pagerank_features(compute_weighted_pagerank_features(otpw_2017_df))
otpw_2018_df = compute_unweighted_pagerank_features(compute_weighted_pagerank_features(otpw_2018_df))
otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)

otpw_test_df = compute_unweighted_pagerank_features(compute_weighted_pagerank_features(otpw_test_df))

# Fill Missing Values

In [0]:
otpw_train_df = otpw_train_df.fillna({
    "origin_pagerank_weighted": 0.0,
    "origin_pagerank_unweighted": 0.0,
    "dest_pagerank_weighted": 0.0,
    "dest_pagerank_unweighted": 0.0
})

otpw_test_df = otpw_test_df.fillna({
    "origin_pagerank_weighted": 0.0,
    "origin_pagerank_unweighted": 0.0,
    "dest_pagerank_weighted": 0.0,
    "dest_pagerank_unweighted": 0.0
})

# Write otpw train and test sets with window-based and PageRank features

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041726_otpw_train_df.parquet")
otpw_test_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041726_otpw_test_df.parquet")
# otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041726_otpw_train_df.parquet")
# otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041726_otpw_test_df.parquet")